# ?곗씠??遺덈윭?ㅺ린

In [ ]:
# sas ?곗씠?곕룄 遺덈윭?????덉쓬
import pandas as pd
df_total = pd.read_sas('./data/hn16_all.sas7bdat', format='sas7bdat') # 嫄닿컯愿???꾩껜 ?곗씠??2016??
df_food = pd.read_sas('./data/hn16_ffq.sas7bdat', format='sas7bdat') # ?앺뭹??랬 ?곗씠??2016??
df_food

In [ ]:
df_total

# ?곗씠???꾩쿂由?


## 吏덈퀝 ?곗씠???꾩쿂由?


In [ ]:
# 吏덈퀝 愿??而щ읆 ?뺤씤
df_total.iloc[:, 52:194]
# 吏덈퀝 愿??而щ읆 以?'dg'媛 ?ㅼ뼱媛?寃껊쭔 異붿텧
df_total.iloc[:, [i for i, x in enumerate(list(df_total.columns)) if 'dg' in x]]

In [ ]:
# 吏덈퀝 愿??而щ읆 以?'dg'媛 ?ㅼ뼱媛?寃껊쭔 異붿텧
# base_columns : 'ID', 'sex', 'age' 異붽?
df_disease = df_total.iloc[:, [1, 8, 9] + [i for i, x in enumerate(list(df_total.columns)) if 'dg' in x]]
df_disease

In [ ]:
# 吏덈퀝 蹂?value_counts ?뺤씤
# dg濡??앸굹硫??섏궗吏꾨떒 ?щ?
for num in [i for i, x in enumerate(list(df_total.columns)) if 'dg' in x]:
    print(df_total.iloc[:, num].value_counts())

## ?뚯떇 ??랬 ?곗씠???꾩쿂由?


In [ ]:
# FF_RICE?먯꽌 寃곗륫移??덈뒗 ??諛?臾댁쓳???쒓굅
df_food = df_food.dropna(subset=['FF_RICE'])
df_food = df_food[df_food['FF_RICE'] != 99]
df_food

In [ ]:
# 1??湲곗? ?곸뼇????랬??而щ읆 異붿텧
df_day_nutr = df_food.loc[:, 'FQ_EN':'FQ_VITC']
df_day_nutr = df_food[['ID'] + list(df_day_nutr.columns)].dropna()
df_day_nutr

## 理쒖쥌 ?ъ슜???곗씠???꾩쿂由?


In [ ]:
# 吏덈퀝 ?곗씠?곗? ?곸뼇???곗씠?곕? 寃고빀
df_final = pd.merge(df_disease, df_day_nutr, on='ID')
df_final

In [ ]:
df_final.columns

# 吏덈퀝 ?덉륫

???: 怨좏삁?? ?댁긽吏吏덊삁利? ?뚯「以? ?ш렐寃쎌깋, ?묒떖利? ?먭껐?? ?밸눊蹂? ?꾩븫

In [ ]:
disease_list = ['怨좏삁??, '?댁긽吏吏덊삁利?, '?뚯「以?, '?ш렐寃쎌깋', '?묒떖利?, '?먭껐??, '?밸눊蹂?, '?꾩븫']
disease_codes = ['DI1_dg', 'DI2_dg', 'DI3_dg', 'DI5_dg', 'DI6_dg', 'DJ2_dg', 'DE1_dg', 'DC1_dg']

In [ ]:
# 吏덈챸 肄붾뱶 - 吏덈퀝 ?대쫫 ?뺤뀛?덈━ ?앹꽦
disease_dict = dict(zip(disease_codes, disease_list))
disease_dict

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn import svm
import xgboost as xgb
import lightgbm as lgb

# ?ъ슜??紐⑤뜽 ?뺤쓽
log_reg = LogisticRegression(random_state=42, n_jobs=-1)
SGD = SGDClassifier(random_state=42, n_jobs=-1)
DTree = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
model_xgb = xgb.XGBClassifier(random_state=42, n_jobs=-1)
model_lgb = lgb.LGBMClassifier(random_state=42, n_jobs=-1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# kfold ?곸슜
from sklearn.model_selection import KFold, StratifiedKFold
import numpy as np

def rmse_cv(model, X, y):
    # cv蹂꾨줈 ?숈뒿?섎뒗 ?⑥닔 - Stratified KFold 援먯감寃利?n    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    acc_scores = []
    recall_scores = []
    precision_scores = []  
    f1_scores = []  

    model_name = model.__class__.__name__
    for train_index, test_index in kf.split(X, y):
        # train, test ?곗씠??遺꾨━
        X_train, X_test = X.iloc[train_index, :], X.iloc[test_index, :]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        # 紐⑤뜽 ?숈뒿 諛??덉륫
        clf = model.fit(X_train, y_train)
        pred = clf.predict(X_test)

        # ?됯? 吏??怨꾩궛

        acc = accuracy_score(y_test, pred)
        precsion = precision_score(y_test, pred)
        recall = recall_score(y_test, pred)
        f1 = f1_score(y_test, pred)
        # confusion_matrix_print(y_test, pred)
        acc_scores.append(acc)
        recall_scores.append(recall)
        precision_scores.append(precsion)
        f1_scores.append(f1)

    return model_name, acc_scores, recall_scores, precision_scores, f1_scores


def print_score(model, X, y):
    # cv蹂??꾨┛?? ?됯퇏 ???n    model_name, acc_scores, recall_scores, precision_scores, f1_scores = rmse_cv(model, X, y)
    n_splits = [i for i in range(1,6)]
    # for i, a, r, p, f  in zip(n_splits, acc_scores, recall_scores, precision_scores, f1_scores):
    #     print(f'{i} FOLDS: {model_name} Acuuracy: {a:.4f}, Recall: {r:.4f}, Precision: {p:.4f}, f1_score: {f:.4f}')
    print(f'\n{model_name} mean Acuuracy: {np.mean(acc_scores):.4f}, mean Recall: {np.mean(recall_scores):.4f}, mean Precision: {np.mean(precision_scores):.4f}, mean f1_score: {np.mean(f1_scores):.4f}')
    print('='*40)
    return model_name, np.mean(acc_scores), np.mean(recall_scores), np.mean(precision_scores), np.mean(f1_scores)

In [ ]:
# 吏덈퀝 蹂??뺤쭊?????뺤씤
for disease_name in disease_codes:
    print(df_final[disease_name].value_counts())

In [ ]:
from imblearn.over_sampling import SMOTE

result_df_dict = dict() # ?깅뒫 ?됯? df ??ν븯???뺤뀛?덈━ ?앹꽦
for disease_name in disease_codes:
        print('吏덈퀝 :' + disease_dict[disease_name])
        select_cols = ['sex', 'age'] + list(df_final.loc[:, 'FQ_EN':'FQ_VITC'].columns)
        df = df_final[select_cols]

        df = df[(df_final[disease_name] == 1) | (df_final[disease_name] == 0)] # 吏덈퀝???덈뒗 ?щ엺怨??녿뒗 ?щ엺留?異붿텧(誘몄쓳???쒓굅)
 
        df[disease_name] = df_final[disease_name] # ?덉륫??吏덈퀝 而щ읆 異붽?

        X = df.iloc[:, :-1] # X,y 遺꾨━
        y = df.iloc[:, -1]

        # 吏덈퀝 鍮꾩쑉
        print(f'?뺤쭊 鍮꾩쑉 : {round(df[disease_name].value_counts()[1]/df.shape[0] * 100, 2)}%')

        # SMOTE ?ㅻ쾭?섑뵆留??섑뻾
        # sm = SMOTE(random_state=42, k_neighbors=2)   
        sm = SMOTE(random_state=42)  
        X_sm, y_sm = sm.fit_resample(X, y)

        # 遺꾨쪟紐⑤뜽 ?숈뒿 諛??됯?
        models = []
        acc_scores = []
        recall_scores = []
        precision_scores = []  
        f1_scores = []  
        for model in [log_reg, SGD, DTree, rf, model_xgb, model_lgb]:
            model_name, mean_acc, mean_recall, mean_precision, mean_f1 = print_score(model, X_sm, y_sm)

            models.append(model_name)
            acc_scores.append(mean_acc)
            recall_scores.append(mean_recall)
            precision_scores.append(mean_precision)
            f1_scores.append(mean_f1)

        result_df = pd.DataFrame({'Model': models, 'Acurracy': acc_scores, 'Recall': recall_scores, 'Precision': precision_scores, 'f1': f1_scores}).reset_index(drop=True)
        result_df_dict[disease_name] = result_df # ?깅뒫 寃곌낵 df ???n        

In [ ]:
result_df_dict['DI1_dg']

In [ ]:
opt_threshold_df = pd.DataFrame(columns=['disease', 'Opt_Threshold'])

for disease_name in disease_codes:
        print(disease_dict[disease_name])
        disease_row = []
        disease_row.append(disease_dict[disease_name])
        select_cols = ['sex', 'age'] + list(df_final.loc[:, 'FQ_EN':'FQ_VITC'].columns)
        df = df_final[(df_final[disease_name] == 1) | (df_final[disease_name] == 0)]
        # df = df[df['sex'] == sex_code]
        df = df[select_cols]
        df[disease_name] = df_final[disease_name]

        X = df.iloc[:, :-1]
        y = df.iloc[:, -1]
        kf = KFold(n_splits=5, shuffle=True, random_state=42)

        # ?쒕뜡 ?щ젅?ㅽ듃 紐⑤뜽 ?앹꽦
        # params = {'n_estimators': 100, 'ax_depth': 5, 'in_samples_split': 2, 'random_state': 42}
        model = xgb.XGBClassifier(random_state=42, n_jobs=-1)

        # 理쒖쟻 ?꾧퀎媛믪쓣 李얘린 ?꾪빐 援먯감 寃利??섑뻾
        best_threshold = 0
        best_score = 0
        precisions = []
        recalls = []
        sm = SMOTE(random_state=42, k_neighbors = 4)  
        X_sm, y_sm = sm.fit_resample(X, y)

        for threshold in np.arange(0.1, 0.98, 0.03):
                # print('?꾧퀎媛?', threshold)
                score = 0
                precision = []
                recall = []
                for train_index, val_index in kf.split(X_sm):
                        X_train, X_val = X_sm.iloc[train_index], X_sm.iloc[val_index]
                        y_train, y_val = y_sm.iloc[train_index], y_sm.iloc[val_index]
                        # ?쒕뜡 ?щ젅?ㅽ듃 紐⑤뜽 ?숈뒿
                        model.fit(X_train, y_train)


                        # 寃利??곗씠?곗뀑?쇰줈 ?덉륫
                        y_pred = model.predict_proba(X_val)[:, 1]

                        # ?꾧퀎媛믪쓣 ?곸슜?섏뿬 ?댁쭊 遺꾨쪟 ?섑뻾
                        y_pred_binary = (y_pred > threshold).astype(int)

                        # ?뺣??? ?ы쁽??怨꾩궛
                        tp = np.sum((y_val == 1) & (y_pred_binary == 1))
                        fp = np.sum((y_val == 0) & (y_pred_binary == 1))
                        fn = np.sum((y_val == 1) & (y_pred_binary == 0))
                        precision.append(tp / (tp + fp))
                        recall.append(tp / (tp + fn))

                        # ?뺥솗??怨꾩궛
                        accuracy = np.mean(y_val == y_pred_binary)
                        score += accuracy

                # 援먯감 寃利?寃곌낵 ?됯퇏 怨꾩궛
                score /= kf.get_n_splits()

                # 理쒖쟻 ?꾧퀎媛??낅뜲?댄듃
                if score > best_score:
                        best_score = score
                        best_threshold = threshold



                # 援먯감 寃利?寃곌낵 ?됯퇏 怨꾩궛
                precision = np.mean(precision)
                recall = np.mean(recall)
                precisions.append(precision)
                recalls.append(recall)
        
        print('理쒖쟻 ?꾧퀎媛?', round(best_threshold, 2))
        opt_threshold_df.loc[len(opt_threshold_df)] = [disease_dict[disease_name], best_threshold]
        # 洹몃옒??洹몃━湲?n        plt.figure(figsize=(7, 4))
        plt.plot(np.arange(0.1, 0.95, 0.03), recalls, label='recall')
        plt.plot(np.arange(0.1, 0.95, 0.03), precisions, label='precision')
        # plt.plot([best_threshold, best_threshold], [0, 1], linestyle='--', label='best threshold')
        plt.xlabel('threshold')
        plt.grid()
        plt.legend()
        plt.show()

In [ ]:
thresholds

In [ ]:
len(precision), len(recall), len(thresholds)

In [ ]:
import numpy as np
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

opt_threshold_df = pd.DataFrame(columns=['disease', 'Opt_Threshold'])

for disease_name in disease_codes:
    print(disease_dict[disease_name])
    select_cols = ['sex', 'age'] + list(df_final.loc[:, 'FQ_EN':'FQ_VITC'].columns)
    df = df_final[(df_final[disease_name] == 1) | (df_final[disease_name] == 0)]
    # df = df[df['sex'] == sex_code]
    df = df[select_cols]
    df[disease_name] = df_final[disease_name]

    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]

    sm = SMOTE(random_state=42, k_neighbors = 4)  
    X_sm, y_sm = sm.fit_resample(X, y)

    X_train, X_val, y_train, y_val = train_test_split(X_sm, y_sm, test_size=0.2, random_state=42)
    
    model = xgb.XGBClassifier(random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    # PR 洹몃옒?꾨? 洹몃━湲??꾪빐 ?꾧퀎媛믪쓣 0.5濡??ㅼ젙
    y_pred = model.predict_proba(X_val)[:, 1]
    precision, recall, thresholds = metrics.precision_recall_curve(y_val, y_pred)
    recall = recall[1:]
    precision = precision[1:]

    # 理쒖쟻???꾧퀎媛?李얘린
    optimal_threshold = thresholds[np.argmin(np.abs(precision + (1 - recall) - 1))]
    opt_threshold_df.loc[len(opt_threshold_df)] = [disease_dict[disease_name], optimal_threshold]
    # 洹몃옒??洹몃━湲?n    plt.figure(figsize=(7, 4))
    plt.plot(thresholds, recall, label='recall')
    plt.plot(thresholds, precision, label='precision')
    plt.plot(optimal_threshold, recall[np.argmin(np.abs(precision + (1 - recall) - 1))], 'ro', label='optimal threshold')
    plt.xlabel('threshold')
    plt.grid()
    plt.legend()
    plt.show()

    print('理쒖쟻???꾧퀎媛?', optimal_threshold)
    print()

In [ ]:
opt_threshold_df

In [ ]:
X

In [ ]:
sample

In [ ]:
# ?쒕챸??吏꾨떒
for i in range(1, 100, 100):
        i = 809
        pred_df_t = pd.DataFrame(columns=['吏덈퀝.', '吏꾨떒 ?뺣쪧', '?덉쟾 ?쇰꺼'])
        sample = df_final.iloc[[i], :]                
        sample = sample[X.columns]
        # sample = sample.iloc[:, 2:]
        df = df_final.drop(df_final.index[i])
        print(i)
        for disease_name in disease_codes:
                disease_row = []
                disease_row.append(disease_dict[disease_name])
                select_cols = list(df_final.loc[:, 'FQ_EN':'FQ_VITC'].columns)
                df = df[select_cols]
                df[disease_name] = df_final[disease_name]
                df = df[(df[disease_name] == 1) | (df[disease_name] == 0)]
                # df = df[df['sex'] == sex_code]
                
                X = df.iloc[:, :-1]
                y = df.iloc[:, -1]
                # i踰덉㎏ ?곗씠???쒓굅
                X = X.drop(X.index[i])
                y = y.drop(X.index[i])


                # SMOTE ?ㅻ쾭?섑뵆留??섑뻾
                #sm = SMOTE(random_state=42, k_neighbors=2)   
                sm = SMOTE(random_state=42, k_neighbors = 4)  
                X_sm, y_sm = sm.fit_resample(X, y)

                model_xgb.fit(X_sm, y_sm)
                pred = model_xgb.predict_proba(sample)
                proba = np.round(pred[0][1], 3)
                disease_row.append(proba)
                if proba >= opt_threshold_df[opt_threshold_df['disease'] == disease_dict[disease_name]]['Opt_Threshold'].values[0]:
                        disease_row.append('怨좎쐞?섍뎔')
                        
                # elif proba >= (opt_threshold_df[opt_threshold_df['disease'] == disease_dict[disease_name]]['Opt_Threshold'].values[0] / 2):
                #         disease_row.append('以묒쐞?섍뎔')
                else:
                        disease_row.append('??꾪뿕援?)
                pred_df_t.loc[len(pred_df_t)] = disease_row
pred_df_t

In [ ]:
sample

In [ ]:
df_final.iloc[[i], :]   

In [ ]:
# 2030 以?吏덈퀝 ?덈뒗 ?좊뱾 戮묒븘???덉떆濡??ъ슜?섎젮怨?. ?몃뜳??戮묒븘 ??n# ?꾩쓽 寃곌낵 戮묐뒗 肄붾뱶?먯꽌 i 媛믩쭔 諛붽씀硫???nfor disease_name in disease_codes:
    temp = df_final[df_final['age'] < 39][[disease_name]]
    print(temp[temp[disease_name] == 1].index)

In [ ]:
df_final

In [ ]:
sample_df = pd.DataFrame(columns=['?깅퀎', '?섏씠'] +  nutri_name)
sample_df

In [ ]:
df_final

In [ ]:
df_final.loc[809, ['sex', 'age']]

In [ ]:
sample_df

In [ ]:
sample_df

In [ ]:
df_final.loc[809, ['sex', 'age']].values

In [ ]:
df_final.loc[[809], 'FQ_EN':].round(2).values[0]

In [ ]:
sample_df = pd.DataFrame(columns=['?깅퀎', '?섏씠'] +  nutri_name)
sample_df.loc[0, :] = list(df_final.loc[809, ['sex', 'age']].values) + list(df_final.loc[[809], 'FQ_EN':].round(2).values[0])
sample_df


In [ ]:
nutri_codes = ['FQ_EN', 'FQ_PROT', 'FQ_FAT', 'FQ_SFA','FQ_MUFA', 'FQ_PUFA', 'FQ_N3', 'FQ_N6', 'FQ_CHOL', 'FQ_CHO', 
                'FQ_TDF', 'FQ_CA', 'FQ_PHOS', 'FQ_FE', 'FQ_NA', 'FQ_K', 'FQ_VA', 'FQ_RETIN', 'FQ_CAROT', 'FQ_B1', 'FQ_B2', 'FQ_NIAC', 'FQ_VITC']
nutri_name = ['?먮꼫吏(kcal)', '?⑤갚吏?g)', '吏諛?g)', '?ы솕吏諛⑹궛(g)', '?⑥씪遺덊룷?붿?諛⑹궛(g)', '?ㅺ?遺덊룷?붿?諛⑹궛(g)', 'n-3怨?g)', 'n-6怨?g)', '肄쒕젅?ㅽ뀒濡?mg)', '?꾩닔?붾Ъ(g)','?앹씠?ъ쑀(g)', '移쇱뒛(mg)', '??mg)', '泥?mg)', '?섑듃瑜?mg)', '移쇰ⅷ(mg)', '鍮꾪?誘퍪(?뛕E)',  '移대줈???뛕E)', '?덊떚?(?뛕E)', '?곗븘誘?mg)', '由щ낫?뚮씪鍮?mg)', '?섏씠?꾩떊(mg)', '鍮꾪?誘퍬(mg)']

In [ ]:
len(nutri_codes)

In [ ]:
nutri_dict = dict(zip(nutri_codes, nutri_name))

In [ ]:
sample

## 怨좏삁?뺣쭔 吏꾪뻾

In [ ]:
# 怨좏삁??ndisease_name= 'DI1_dg'
select_cols = ['sex'] + list(df_19_39.loc[:, 'FQ_EN':'FQ_VITC'].columns)
df_gha = df_19_39[(df_19_39[disease_name] == 1) | (df_19_39[disease_name] == 0)]
df_gha = df_gha[select_cols]
df_gha[disease_name] = df_19_39[disease_name]
df_gha

##### Stratified K-Fold ?ъ슜?댁꽌 ?덉륫 諛?援먯감寃利?


In [ ]:
X = df_gha_male.iloc[:, :-1]
y = df_gha_male.iloc[:, -1]
X.shape, y.shape

In [ ]:
# 怨좏삁??鍮꾩쑉
print(df_gha_male[disease_name].value_counts()[0]/df_gha_male.shape[0])

##### SMOTE ?ㅻ쾭?섑뵆留??섑뻾

In [ ]:
from imblearn.over_sampling import SMOTE

# SMOTE ?ㅻ쾭?섑뵆留??섑뻾
sm = SMOTE(random_state=42)
X_sm, y_sm = sm.fit_resample(X, y)
X_sm.shape, y_sm.shape

In [ ]:
models = []
acc_scores = []
recall_scores = []
precision_scores = []  
f1_scores = []  
for model in [log_reg, SGD, DTree, rf, model_xgb, model_lgb]:
    model_name, mean_acc, mean_recall, mean_precision, mean_f1 = print_score(model, X_sm, y_sm)
    models.append(model_name)

    acc_scores.append(mean_acc)
    recall_scores.append(mean_recall)
    precision_scores.append(mean_precision)
    f1_scores.append(mean_f1)
result_df = pd.DataFrame({'Model': models, 'Acurracy': acc_scores, 'Recall': recall_scores, 'Precision': precision_scores, 'f1': f1_scores}).reset_index(drop=True)
result_df

In [ ]:
# XGBClassifier feature importance ?쒓컖??n
model_xgb.fit(X_sm, y_sm)
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(10, 10))
sns.barplot(x=model_xgb.feature_importances_, y=X.columns)
plt.show()

In [ ]:
df_1 = pd.read_excel('./data/?곸뼇??蹂???좎젙?.xlsx')
df_2 = pd.read_excel('./data/?곸뼇??蹂???댄쁽吏.xlsx')

In [ ]:
df_nutrient = pd.concat([df_1, df_2])
df_nutrient.reset_index(drop=True, inplace=True)
df_nutrient

### ?섏씠 ?쒖젙 O

In [ ]:
df_gha['age'].describe()

In [ ]:
df_gha_20_30 = df_gha[df_gha['age'] < 40] 
df_gha_20_30

In [ ]:
df_gha_20_30 = df_gha[df_gha['age'] < 40] 
df_gha_20_30[disease_name].value_counts()